<a href="https://colab.research.google.com/github/Flguima/PROJETOS/blob/main/PREVIS%C3%83O_OCORR%C3%8ANCIA_INC%C3%8ANCIO_AMBIENTAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cross Validation**

Nesta tarefa, trabalhei com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [ ]:
# ================================
# Importação das bibliotecas necessárias
# ================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print("Eu importei pandas, numpy e StandardScaler para manipular e preparar os dados.")

# ================================
# Carregamento da base de dados
# ================================
# Eu carrego o arquivo diretamente do Desktop do usuário USER
df = pd.read_csv(r"C:\Users\USER\Desktop\smoke_detection_iot.csv")

print("Eu carreguei a base de dados!")
print("Visualizando as primeiras linhas para confirmar:")
print(df.head())

# ================================
# Ajuste do nome da coluna alvo
# ================================
# Eu renomeio a coluna 'Fire Alarm' para 'Fire_Alarm' para evitar problemas com espaços.
df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

print("Eu renomeei a coluna alvo para Fire_Alarm!")

# ================================
# Remoção de colunas irrelevantes (checando antes)
# ================================
# Algumas bases exportadas trazem colunas como 'Unnamed:0' ou 'UTC'.
# Eu verifico se elas existem antes de remover, para evitar erros.
colunas_irrelevantes = [col for col in ['Unnamed:0', 'UTC'] if col in df.columns]

if colunas_irrelevantes:
    df.drop(columns=colunas_irrelevantes, inplace=True)
    print(f"Eu removi as colunas irrelevantes: {colunas_irrelevantes}")
else:
    print("Não havia colunas irrelevantes para remover.")

# ================================
# Verificação dos tipos de dados
# ================================
print("Eu verifiquei os tipos de dados de cada coluna:")
print(df.dtypes)

# ================================
# Checagem de valores nulos
# ================================
print("Eu verifiquei se há valores nulos na base:")
print(df.isnull().sum())

# ================================
# Tratamento de valores nulos
# ================================
# Se houver valores nulos, eu imputo a média para variáveis numéricas.
df.fillna(df.mean(), inplace=True)

print("Eu tratei os valores nulos imputando a média nas colunas numéricas.")

# ================================
# Normalização dos dados
# ================================
# Eu normalizo os dados numéricos para que fiquem na mesma escala,
# o que ajuda os algoritmos de classificação a performarem melhor.
features = df.drop(columns=['Fire_Alarm'])
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)

# Eu junto novamente a variável alvo
df_scaled['Fire_Alarm'] = df['Fire_Alarm']

print("Eu normalizei os dados numéricos para melhorar a performance do modelo.")

# ================================
# Conclusão inicial
# ================================
print("Eu finalizei o tratamento da base: sem colunas irrelevantes, sem valores nulos e com variáveis normalizadas.")

Eu importei pandas, numpy e StandardScaler para manipular e preparar os dados.
Eu carreguei a base de dados!
Visualizando as primeiras linhas para confirmar:
   Unnamed: 0         UTC  Temperature[C]  Humidity[%]  TVOC[ppb]  eCO2[ppm]  \
0           0  1654733331          20.000        57.36          0        400   
1           1  1654733332          20.015        56.67          0        400   
2           2  1654733333          20.029        55.96          0        400   
3           3  1654733334          20.044        55.28          0        400   
4           4  1654733335          20.059        54.69          0        400   

   Raw H2  Raw Ethanol  Pressure[hPa]  PM1.0  PM2.5  NC0.5  NC1.0  NC2.5  CNT  \
0   12306        18520        939.735    0.0    0.0    0.0    0.0    0.0    0   
1   12345        18651        939.744    0.0    0.0    0.0    0.0    0.0    1   
2   12374        18764        939.738    0.0    0.0    0.0    0.0    0.0    2   
3   12390        18849        939.736

Para a coluna Fire Alarm, por conta do espaçamento talvez seja util renomear o nome da coluna utilizando:

df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.




Eu escolho regressão logística porque:
A variável Fire_Alarm é binária (0 ou 1), e esse modelo é ideal para classificação binária.
É eficiente e rápido de treinar, o que é essencial em aplicações IoT que precisam de resposta em tempo real.
É interpretável: consigo analisar os coeficientes e entender quais variáveis ambientais (temperatura, umidade, CO2, partículas etc.) mais influenciam na probabilidade de incêndio. Funciona muito bem com validação cruzada, permitindo avaliar estabilidade e generalização do modelo. É um ponto de partida sólido antes de comparar com modelos mais complexos (como Random Forest ou Gradient Boosting).


# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

In [ ]:
# Separação em X e y
# ================================
X = df.drop(columns=['Fire_Alarm'])   # Variáveis independentes (sensores ambientais)
y = df['Fire_Alarm']                  # Variável alvo

print("Separei a base em X (preditores) e y (alvo Fire_Alarm).")

# ================================
# 7. Normalização dos dados
# ================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Normalizei os dados para que fiquem na mesma escala.")

# ================================
# 8. Escolha e instância do modelo
# ================================
# Justificativa:
# - O Fire_Alarm é binário (0 ou 1), e a regressão logística é ideal para classificação binária.
# - É eficiente e rápido de treinar, essencial em aplicações IoT.
# - É interpretável: permite entender quais variáveis ambientais mais influenciam na probabilidade de incêndio.
# - Funciona muito bem com validação cruzada, garantindo avaliação justa e robusta.
model = LogisticRegression(max_iter=1000)

print("Instanciei o modelo de regressão logística para prever Fire_Alarm.")

# ================================
# 9. Validação cruzada K-Fold
# ================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')

print("Rodei o modelo com validação cruzada K-Fold.")
print("Acurácias obtidas em cada fold:", scores)
print("Média da acurácia:", scores.mean())

Separei a base em X (preditores) e y (alvo Fire_Alarm).
Normalizei os dados para que fiquem na mesma escala.
Instanciei o modelo de regressão logística para prever Fire_Alarm.
Rodei o modelo com validação cruzada K-Fold.
Acurácias obtidas em cada fold: [0.98834424 0.98690723 0.9863484  0.98714673 0.98531055]
Média da acurácia: 0.9868114322209804


# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [ ]:
# ================================
# Definição do número de folds
# ================================
# Eu escolho 5 folds porque é um valor padrão que equilibra bem
# entre tempo de execução e robustez estatística.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Defini a validação cruzada com 5 folds.")

# ================================
# Rodando o modelo com validação cruzada
# ================================
# Eu uso cross_val_score para avaliar o modelo em cada fold.
scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')

print("Rodei o modelo de regressão logística com validação cruzada K-Fold.")
print("Acurácias obtidas em cada fold:", scores)
print("Média da acurácia:", scores.mean())

Defini a validação cruzada com 5 folds.
Rodei o modelo de regressão logística com validação cruzada K-Fold.
Acurácias obtidas em cada fold: [0.98834424 0.98690723 0.9863484  0.98714673 0.98531055]
Média da acurácia: 0.9868114322209804


# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [ ]:
# ================================
# Definição do número de folds
# ================================
from sklearn.model_selection import KFold, cross_val_score

# Eu escolho 5 folds porque é um valor padrão que equilibra bem
# entre tempo de execução e robustez estatística.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Defini a validação cruzada com 5 folds.")

# ================================
# Rodando o modelo com validação cruzada
# ================================
# Eu uso cross_val_score para avaliar o modelo em cada fold.
scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='accuracy')

# ================================
# Avaliação dos resultados
# ================================
print("Resultados da validação cruzada K-Fold:")
for i, score in enumerate(scores, start=1):
    print(f"Fold {i}: acurácia = {score:.4f}")

print("Validação final - média da acurácia:", scores.mean())

Defini a validação cruzada com 5 folds.
Resultados da validação cruzada K-Fold:
Fold 1: acurácia = 0.9883
Fold 2: acurácia = 0.9869
Fold 3: acurácia = 0.9863
Fold 4: acurácia = 0.9871
Fold 5: acurácia = 0.9853
Validação final - média da acurácia: 0.9868114322209804
